# LOB Snapshot Viewer
Ingests `data/MD_small/lob.csv` into `LimitOrderBook` and prints depth snapshots.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from lob import LimitOrderBook
from lob_reader import LOBReader, Snapshot

In [ ]:
DATA = ROOT / "data" / "MD_small" / "lob.csv"

## Snapshot plotter

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def plot_snapshot(
    ax,
    bid_depth: dict,
    ask_depth: dict,
    snap: Snapshot,
    idx: int,
    n_levels: int = 5,
) -> None:
    """Draw one LOB snapshot as a back-to-back horizontal bar chart.

    Bids extend left (green), asks extend right (red).
    Price is on the y-axis; best bid/ask are closest to the centre line.
    Each bar is annotated with the exact size.
    """
    ask_levels = sorted(ask_depth.items())[:n_levels]  # best ask first (lowest price)
    bid_levels = sorted(bid_depth.items(), reverse=True)[
        :n_levels
    ]  # best bid first (highest price)

    all_prices = [p for p, _ in bid_levels] + [p for p, _ in ask_levels]
    all_sizes = [s for _, s in bid_levels] + [s for _, s in ask_levels]

    if not all_prices:
        ax.set_visible(False)
        return

    # Bar height: ~70 % of the smallest price gap so bars don't touch.
    prices_sorted = sorted(set(all_prices))
    if len(prices_sorted) > 1:
        gaps = [
            prices_sorted[i + 1] - prices_sorted[i]
            for i in range(len(prices_sorted) - 1)
        ]
        bar_h = min(gaps) * 0.7
    else:
        bar_h = prices_sorted[0] * 1e-5

    size_scale = max(all_sizes) if all_sizes else 1

    bid_color = "#2ca85a"
    ask_color = "#e04545"

    # ── bids (left side, negative x) ─────────────────────────────────────────
    for price, size in bid_levels:
        ax.barh(price, -size, height=bar_h, color=bid_color, alpha=0.85, linewidth=0)
        ax.text(
            -size,
            price,
            f"{size:,.0f} ",
            ha="right",
            va="center",
            fontsize=7,
            color="white",
            fontweight="bold",
        )
        ax.text(
            size_scale * 0.03,
            price,
            f"{price:.7f}",
            ha="left",
            va="center",
            fontsize=7,
            color=bid_color,
        )

    # ── asks (right side, positive x) ────────────────────────────────────────
    for price, size in ask_levels:
        ax.barh(price, size, height=bar_h, color=ask_color, alpha=0.85, linewidth=0)
        ax.text(
            size,
            price,
            f" {size:,.0f}",
            ha="left",
            va="center",
            fontsize=7,
            color="white",
            fontweight="bold",
        )
        ax.text(
            -size_scale * 0.03,
            price,
            f"{price:.7f}",
            ha="right",
            va="center",
            fontsize=7,
            color=ask_color,
        )

    # ── centre line ───────────────────────────────────────────────────────────
    ax.axvline(0, color="#888", linewidth=0.8, zorder=3)

    # ── mid / spread indicator ────────────────────────────────────────────────
    best_bid = bid_levels[0][0] if bid_levels else None
    best_ask = ask_levels[0][0] if ask_levels else None
    if best_bid and best_ask:
        mid = (best_bid + best_ask) / 2
        sprd = best_ask - best_bid
        ax.axhline(mid, color="#888", linewidth=0.6, linestyle="--", zorder=2)

    # ── axes formatting ───────────────────────────────────────────────────────
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):,.0f}"))
    ax.set_yticks([])  # prices shown as inline text labels
    ax.set_xlabel("size", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)
    ax.set_xlim(-size_scale * 1.45, size_scale * 1.45)

    ts_s = snap.timestamp / 1e6
    title_lines = [f"snapshot #{idx}   t={ts_s:.3f} s"]
    if best_bid and best_ask:
        title_lines.append(f"mid={mid:.7f}   spread={sprd:.7f}")
    ax.set_title("\n".join(title_lines), fontsize=8, pad=4)

    # ── legend (only on first subplot) ───────────────────────────────────────
    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(color=bid_color, label="bid"),
            Patch(color=ask_color, label="ask"),
        ],
        fontsize=7,
        loc="upper right",
        framealpha=0.5,
    )

## Ingest first N snapshots and print every K-th

In [ ]:
N_SNAPSHOTS = 50  # how many rows to ingest
PRINT_EVERY = 10  # capture a snapshot every K rows
N_LEVELS = 5  # price levels per side to show

lob = LimitOrderBook()
reader = LOBReader(lob)

# Collect (idx, snap, bid_depth, ask_depth) at each print point during ingestion.
captured = []

for idx, (snap, _) in enumerate(reader.ingest_csv(DATA)):
    if idx >= N_SNAPSHOTS:
        break
    if idx % PRINT_EVERY == 0:
        captured.append((idx, snap, dict(lob.bid_depth()), dict(lob.ask_depth())))

# ── plot grid ──────────────────────────────────────────────────────────────────
n = len(captured)
cols = min(n, 3)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.5, rows * 4.5))
axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

for ax, (idx, snap, bid_d, ask_d) in zip(axes_flat, captured):
    plot_snapshot(ax, bid_d, ask_d, snap, idx, n_levels=N_LEVELS)

# Hide unused axes
for ax in list(axes_flat)[n:]:
    ax.set_visible(False)

fig.suptitle("LOB depth snapshots", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## Show what changed between two consecutive snapshots

In [ ]:
import csv
from lob_reader import Snapshot


def load_snapshots(path, n=5):
    snaps = []
    with open(path) as fh:
        for row in csv.DictReader(fh):
            snaps.append(Snapshot.from_row(row))
            if len(snaps) == n:
                break
    return snaps


snaps = load_snapshots(DATA, n=6)


def diff_snapshots(prev: Snapshot, cur: Snapshot):
    """Return human-readable list of changes."""
    changes = []
    for side_label, prev_d, cur_d in [
        ("BID", prev.bids, cur.bids),
        ("ASK", prev.asks, cur.asks),
    ]:
        all_prices = set(prev_d) | set(cur_d)
        for p in sorted(all_prices):
            old = prev_d.get(p)
            new = cur_d.get(p)
            if old == new:
                continue
            if old is None:
                changes.append(f"  {side_label} +{p:.7f}  size={new:,.0f}  (new level)")
            elif new is None:
                changes.append(f"  {side_label}  {p:.7f}  size={old:,.0f}  → removed")
            else:
                delta = new - old
                sign = "+" if delta > 0 else ""
                changes.append(
                    f"  {side_label}  {p:.7f}  {old:,.0f} → {new:,.0f}  ({sign}{delta:,.0f})"
                )
    return changes


for i in range(1, len(snaps)):
    prev, cur = snaps[i - 1], snaps[i]
    changes = diff_snapshots(prev, cur)
    dt_us = cur.timestamp - prev.timestamp
    print(f"── snapshot {i - 1} → {i}  (Δt={dt_us} µs) ─────────────────────")
    for c in changes:
        print(c)
    print()

## Cumulative depth chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Re-ingest from scratch so we capture final state of a single snapshot.
lob2 = LimitOrderBook()
reader2 = LOBReader(lob2)

target = 20  # which snapshot to visualise
for idx, (snap, _) in enumerate(reader2.ingest_csv(DATA)):
    if idx == target:
        snap_vis = snap
        break

ask_d = sorted(lob2.ask_depth().items())[:25]
bid_d = sorted(lob2.bid_depth().items(), reverse=True)[:25]

bid_prices, bid_sizes = zip(*bid_d)
ask_prices, ask_sizes = zip(*ask_d)

bid_cum = np.cumsum(bid_sizes)
ask_cum = np.cumsum(ask_sizes)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# left: bar chart of top 10 levels
ax = axes[0]
n = 10
ax.barh(
    [f"{p:.7f}" for p in list(reversed(ask_prices[:n]))],
    list(reversed(ask_sizes[:n])),
    color="#e05252",
    label="ask",
)
ax.barh(
    [f"{p:.7f}" for p in bid_prices[:n]], bid_sizes[:n], color="#52a852", label="bid"
)
ax.set_xlabel("size")
ax.set_title(f"Top-10 depth  (snapshot #{target})")
ax.legend()

# right: cumulative depth
ax2 = axes[1]
ax2.step(
    list(reversed(bid_prices)), bid_cum, where="post", color="#52a852", label="bid"
)
ax2.step(ask_prices, ask_cum, where="post", color="#e05252", label="ask")
ax2.axvline(lob2.mid_price(), color="grey", linestyle="--", linewidth=0.8, label="mid")
ax2.set_xlabel("price")
ax2.set_ylabel("cumulative size")
ax2.set_title("Cumulative depth")
ax2.legend()

plt.tight_layout()
plt.show()

## Mid-price over time

In [ ]:
lob3 = LimitOrderBook()
reader3 = LOBReader(lob3)

N = 500
timestamps, mids, spreads = [], [], []

for idx, (snap, _) in enumerate(reader3.ingest_csv(DATA)):
    if idx >= N:
        break
    mid = lob3.mid_price()
    spd = lob3.spread()
    if mid is not None:
        timestamps.append(snap.timestamp / 1e6)
        mids.append(mid)
        spreads.append(spd)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

ax1.plot(timestamps, mids, linewidth=0.8, color="steelblue")
ax1.set_ylabel("mid price")
ax1.set_title(f"Mid-price over first {N} snapshots")

ax2.plot(timestamps, spreads, linewidth=0.6, color="orange")
ax2.set_ylabel("spread")
ax2.set_xlabel("time (s)")

plt.tight_layout()
plt.show()